# Prototype recurrence beyond classical ECG interpretation

ProtoSSL learns a bank of 1,000 ECG prototypes on HEEDB without labels. Each downstream
dataset then picks some of them for each of its labels. `proto-recur-multilabel.ipynb`
showed that PTB-XL, CinC and CODE-15% pick the **same** prototypes for the same
rhythm and conduction labels. This notebook asks the same question for diseases that
are **not** read off the ECG. It then asks whether the shared prototypes show
anything about how those diseases appear on the ECG beyond the findings we already
know about.

## Summary

All numbers are from the cells below at `SEED = 42` on the archived runs. If the
notebook is rerun on new results, update this summary by hand.

**Question.** EchoNext labels come from echocardiograms, MIMIC-IV-ECG labels from
discharge diagnoses (phecodes). Two disease pairs match across them:

| Concept | EchoNext (echo) | MIMIC (ICD-10 via phecode 1.2) |
|---|---|---|
| LV systolic dysfunction | `LVEF Lo` (LVEF ≤ 45%) | HFrEF (phecode 428.3) |
| Aortic valve disease | `AS` (moderate or worse stenosis) | nonrheumatic aortic valve disorders (phecode 395.2) |

If both datasets pick the same prototypes for a disease, independently and from
different label sources, the prototypes encode the disease itself rather than a
dataset artifact. If those prototypes also predict the disease beyond the known ECG
findings, they're candidates for something new to read off the ECG.

**Design.**

1. Count shared prototypes per label pair, against every other EchoNext × MIMIC
   pair and across 5 seeds.
2. Read each shared prototype through the label PTB-XL, CinC and CODE-15% gave it.
3. Test, on the test split, whether shared prototypes predict the disease beyond a
   model of known findings: demographics, intervals (splines), MIMIC's machine-read
   statements, and PTB-XL's statements imputed on every ECG. Also test in ECGs with
   no known finding ("normal-read").
4. Replicate the resulting candidates once on the val split, with criteria fixed
   (and committed) before the val results existed.
5. Draw average beats of the candidates for a blinded cardiologist reading.
6. Post hoc, retest the candidates against graded known findings, including ones from
   ECGFounder, a model outside ProtoSSL.

**Results.**

- **Recurrence is real for both diseases.** `AS` ~ aortic valve disorders shares 6.4
  of 14 prototypes and `LVEF Lo` ~ HFrEF 5.8 (means over seeds). They are the top
  two of all 552 EchoNext × MIMIC pairs (median 0, chance 0.2), on par with RBBB
  across PTB-XL and CODE-15% (5.6).
- **The shared prototypes mean known things.** In PTB-XL, LV dysfunction's shared
  prototypes are old MI (anteroseptal, inferolateral, anterolateral, and ST-T changes
  of aneurysm) and incomplete LBBB.
  Aortic valve disease's are conduction (prolonged PR, bundle branch block) and LVH.
  The shared prototypes are enriched for these PTB-XL classes over the whole bank.
- **One prototype goes beyond known findings, for LV systolic dysfunction only.** On
  test, shared prototypes add ΔAUROC +0.027 (EchoNext) and +0.009 (MIMIC) over known
  findings. MIMIC is the dataset that tells candidates apart: EchoNext has no machine
  reads, and there most shared prototypes pass, known-pattern controls included. In
  MIMIC, **991** (PTB-XL: incomplete LBBB) survives adjustment (OR 1.67 per SD, Holm
  p < 0.001). **850** (PTB-XL: anterolateral MI) missed the criterion with spline
  terms (OR 1.35, Holm p 0.07) and was carried forward to val as borderline. Nothing
  survives for aortic valve disease.
- **Both replicate on val** under the prespecified analysis. Adjusted OR per SD: 850
  is 1.31 in EchoNext and 1.23 in MIMIC, 991 is 1.31 and 1.24 (Holm p 0.036 in MIMIC
  for both, over the 4 prespecified tests).
- **Post hoc: graded known findings don't explain 991, but mostly explain 850.**
  Adding graded scores for conduction delay and low voltage, from PTB-XL's probe or
  from ECGFounder, leaves 991's MIMIC val OR at 1.21–1.22 (p ≈ 0.03). With all 150 of
  ECGFounder's statements it is 1.18 (p 0.064), or 1.15 (p 0.11) with the reads too.
  850's falls to 1.07 with PTB-XL's graded conduction/voltage scores. The
  ProtoSSL-imputed probabilities can't decide this: they rebuild about 90% of any
  prototype's similarity, and on val they pull every prototype, controls included, to
  an OR of 0.95–1.16.
- **What they look like** (normal-read val ECGs vs matched ones). 991's ECGs lack the
  small negative deflection after the R wave in II, aVF and V4–V6 in both datasets,
  and have lower lateral R waves. In EchoNext their QRS is also more positive in
  III/aVF, with a smaller R in aVL. 850's pattern differs between the datasets: its
  aVL R is smaller in EchoNext and taller in MIMIC, and in EchoNext its QRS is lower
  in nearly every lead.

**Caveats.**

- **Effects are small:** about +0.01 AUROC on val, and +0.005 over all of
  ECGFounder's statements in MIMIC. MIMIC's effects shrank from test to val, as
  expected for candidates selected on test, and MIMIC val (275 HFrEF positives) can't
  tell an OR of 1.24 from 1.15.
- **The known-findings list is provisional,** pending review by a cardiologist. Some
  present/absent imputed reads miss most of their finding: imputed incomplete LBBB
  flags 0.4% of MIMIC ECGs and catches 4% of the machine-read ones (train). Low
  voltage is recovered at AUROC 0.80, ST-T and left atrial changes at about 0.68. Poor
  R-wave progression has no PTB-XL code, so it's adjusted for in MIMIC only (machine
  read), and in no imputation. EchoNext has no QRS axis measurement; only
  ECGFounder's axis statements cover it there. The post hoc section adds graded
  scores for all of these except R-wave progression.
- **Normal-read evidence is EchoNext's alone, and doesn't single out the
  candidates.** MIMIC's normal-read subset has almost no positives. In EchoNext the
  known-pattern controls 217, 522, 578 and 617 pass the normal-read criterion too.
  This shows the shared prototypes see more than recorded findings, not that the
  candidates differ from the controls.
- **Average beats** are matched on sex, age, heart rate and QRS duration, but not PR
  (991's high-activation ECGs run 8–12 ms longer). Beats are aligned on the largest
  summed amplitude at 100 Hz, so jitter can blunt averaged peaks. Amplitude claims
  need per-ECG measurements.
- **Uncertainty.** The ORs use patient-clustered SEs, but the ΔAUROC bootstrap
  resamples ECGs, so its CIs may be too narrow. The test ΔAUROC (7 prototypes) and
  the val ΔAUROC (2) aren't comparable as shrinkage.
- **Single seed.** The candidate analysis uses seed 42. Seeds share the prototype
  bank and the test set, so other seeds mainly retest the PTB-XL probe. They haven't
  been run.

## How to read this notebook

| Section | What it shows |
|---|---|
| Datasets and labels | the label mapping and the datasets involved |
| Screen every EchoNext × MIMIC label pair | these two pairs are outliers, not picked by hand |
| Stability across seeds, Overlap at `SEED` | the shared prototypes don't depend on the seed |
| What the shared prototypes mean in classical datasets | PTB-XL/CinC/CODE-15% readings and class enrichment |
| Case studies: projected windows only | single examples of a prototype across datasets |
| Beyond known ECG findings | the incremental-value tests; read its design note first |
| Confirmatory replication on val | the prespecified replication of the candidates |
| Average beats for review | the figures for the cardiologist, labeled and blinded |
| Exploratory: known findings from outside ProtoSSL | post hoc, the candidates against graded known findings |

The loading and validation cells between these sections can be skimmed; they're
there so the analysis fails loudly if an input is misaligned.

## Glossary

- **HEEDB**: Harvard-Emory ECG Database (8.1M ECGs in its train split), used unlabeled
  for pretraining.
- **Prototype / bank**: a learned 1-second, 12-lead ECG pattern; the bank holds 1,000
  of them, learned once on HEEDB.
- **Similarity (activation)**: how strongly an ECG's best-matching 1-second window
  resembles a prototype; each ECG gets one value per prototype.
- **Assignment (ILP)**: per downstream dataset, an integer linear program gives each
  label `PPL` = 14 prototypes from the bank, by effect size on train, using each
  prototype at most once per dataset. A **slot** is one of a label's 14 positions.
- **Shared / recurrent**: assigned to the matching label by more than one dataset.
- **Projection**: replacing a prototype by its most similar real training window, so
  it can be shown as an ECG; each run projects onto its own data.
- **PILA** (`protossl-heedb-pila`): "project in linear assignment", the transfer
  method used here (bank + ILP assignment + projection + linear probe), without
  fine-tuning; `-ft` is the fine-tuned variant.
- **Interpreters**: PTB-XL, CinC and CODE-15%, used only to name what a prototype is.
- **Known findings**: what a reader already uses: demographics, intervals, and
  statements (MIMIC machine reads; PTB-XL statements imputed by PTB-XL's PILA linear
  probe). **+ imputed reads** uses the imputed statements as present/absent, the
  main variant; **+ imputed probabilities** uses them as graded log-odds, which
  absorb nearly any prototype.
- **ECGFounder**: an ECG foundation model trained to predict HEEDB's machine-read
  statements. Its statement head gives graded known findings from outside ProtoSSL
  (post hoc section only).
- **Normal-read**: an ECG with no known finding and normal intervals.
- **Test / val**: held-out splits; nothing in PILA uses either. Candidates were
  chosen on test and confirmed on val.

In [ ]:
import os
os.environ["CACHE_DIR"] = "/home/songs1/.cache/protossl_cache" # for working in dev container

In [ ]:
import collections
import itertools
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from protossl.datasets import (
    CincECGDataset,
    Code15ECGDataset,
    EchoNextECGDataset,
    HeedbECGDataset,
    MimicECGDataset,
    PtbxlECGDataset,
)
from protossl.defines import (
    CINC_TARGETS,
    CODE15_TARGETS,
    ECHONEXT_TARGETS,
    HEEDB_TARGETS,
    MIMIC_TARGETS,
    PTBXL_TARGETS,
)
from protossl.plotting import (
    chunk_to_seconds,
    get_ecg_waveform,
    plot_segment,
    resolve_index,
)

SEED = 42  # seed for the Venn diagrams, interpretation table and case studies
SEEDS = [42, 67, 70, 73, 99]  # all archived seeds, for the screen and stability

# results root: our archive of the paper's runs, or a new run's outputs, e.g.
# /shared/gpudata/songs1/protossl-shared/runs/<run> (see scripts/README.md)
RUN_DIR = Path(os.environ.get("PROTOSSL_OUTPUTS", "~/protossl-outputs")).expanduser() / "ecg"
PRETRAIN_DIR = RUN_DIR / "protossl-heedb"
MODEL = "protossl-heedb-pila"
PPL = 14

DATA_DIR = Path("/opt/gpudata/ecg")
SAMPLING_RATE = 100
SPLIT = "train"

# the HEEDB row of the case studies needs the HEEDB labels cache, which has to be
# built outside the dev container (its cache dir is read-only); turn on once it exists
INCLUDE_PRETRAIN = False

TEXT_WIDTH = 5.5  # inches, width of the paper's text block

FIG_DIR = Path("figs")
FIG_DIR.mkdir(parents=True, exist_ok=True)

## Datasets and labels

`DATASETS` holds the two datasets whose labels we compare. `INTERPRETERS` holds the
classical datasets, used only to read off what a prototype means. `PRETRAIN` is the
HEEDB bank that every dataset draws its prototypes from.

`LABELS` maps a canonical concept onto each dataset's exact target string; the next
cell checks every string against `protossl.defines`. These labels have no HEEDB
pretraining target, which is why there is no HEEDB column. Two mappings are judgement
calls worth knowing about:

- **Aortic valve disease**: phecode 395.2 covers aortic stenosis, regurgitation and
  unspecified aortic valve disorders, so the MIMIC label is broader than EchoNext's
  moderate-or-worse stenosis. EchoNext also has `AR`, which overlaps MIMIC far less.
- **LV systolic dysfunction**: MIMIC also has `Congestive heart failure (chf) nos`
  and `Primary/intrinsic cardiomyopathies`. Both overlap `LVEF Lo` less than HFrEF
  does (see the screen below), as expected for less specific codes.

In [ ]:
PRETRAIN = {
    "slug": "heedb",
    "dataset_cls": HeedbECGDataset,
    "dataset_path": DATA_DIR / "heedb",
    "targets": list(HEEDB_TARGETS),
}

DATASETS = {
    "EchoNext": {
        "slug": "echonext",
        "dataset_cls": EchoNextECGDataset,
        "dataset_path": DATA_DIR / "echonext",
        "targets": list(ECHONEXT_TARGETS),
    },
    "MIMIC": {
        "slug": "mimic",
        "dataset_cls": MimicECGDataset,
        "dataset_path": DATA_DIR / "mimic-iv-ecg",
        "targets": MIMIC_TARGETS,
    },
}

INTERPRETERS = {
    "PTB-XL": {
        "slug": "ptbxl",
        "dataset_cls": PtbxlECGDataset,
        "dataset_path": DATA_DIR / "ptb-xl",
        "targets": PTBXL_TARGETS,
    },
    "CinC": {
        "slug": "cinc",
        "dataset_cls": CincECGDataset,
        "dataset_path": DATA_DIR / "cinc-2020",
        "targets": CINC_TARGETS,
    },
    "CODE-15%": {
        "slug": "code15",
        "dataset_cls": Code15ECGDataset,
        "dataset_path": DATA_DIR / "code15",
        "targets": CODE15_TARGETS,
    },
}

# canonical label -> that label's exact target string in each label set
LABELS = {
    "LV systolic dysfunction": {
        "EchoNext": "LVEF Lo",
        "MIMIC": "Heart failure with reduced ef [systolic or combined heart failure]",
    },
    "Aortic valve disease": {
        "EchoNext": "AS",
        "MIMIC": "Nonrheumatic aortic valve disorders",
    },
}

# filename-safe stems for saved figures
LABEL_SLUGS = {
    "LV systolic dysfunction": "lvsd",
    "Aortic valve disease": "avd",
}

In [ ]:
# every mapped name must exist in the label set it refers to, otherwise a typo
# silently turns into an empty prototype set downstream
for label, mapping in LABELS.items():
    assert label in LABEL_SLUGS, f"no filename slug for {label}"
    assert set(mapping) == set(DATASETS), f"{label}: map every dataset"
    for name, cfg in DATASETS.items():
        assert mapping[name] in cfg["targets"], f"{label}: bad {name} target"

pd.DataFrame(LABELS).T

## Load assignment and projection metadata

- `assignment_metadata.csv` (one row per label slot) says which pretrained prototype
  the ILP gave to each label slot of a dataset. We load it for every seed, since
  the screen and the stability check compare seeds.
- `projection_metadata.csv` (one row per prototype) says which sample and 1-second
  window each prototype was projected onto. We load it only at `SEED`, for plotting.

All seeds share the one HEEDB pretraining run, so prototype indices mean the same
thing in every seed.

In [ ]:
def read_metadata(path: Path) -> pd.DataFrame:
    """Read a metadata CSV, normalizing the sample ID column name.

    `ecg_id` was renamed to `sample_id` when the datasets were generalized beyond
    ECG (see `protossl.trainer.PredictionWriter`), so runs written before that
    rename still carry the old column name. Same with `patient_id` --> `source_id.
    """
    return pd.read_csv(path).rename(columns={"ecg_id": "sample_id", "patient_id": "source_id"})


def run_dir(cfg: dict, seed: int, model: str = MODEL) -> Path:
    return RUN_DIR / f"protossl-outputs-seed{seed}" / f"runs-{cfg['slug']}" / model


heedb_proj = read_metadata(
    PRETRAIN_DIR / "project-prototypes/latest/projection_metadata.csv"
)
N_PROTOTYPES = len(heedb_proj)

ALL_DATASETS = {**DATASETS, **INTERPRETERS}
_datasets = {**ALL_DATASETS, "HEEDB": PRETRAIN}
_dataset_cache: dict[tuple[str, str], object] = {}


def get_dataset(name: str, split: str = SPLIT):
    """Build a dataset once and reuse it, loading waveforms is expensive."""
    if (name, split) not in _dataset_cache:
        cfg = _datasets[name]
        _dataset_cache[(name, split)] = cfg["dataset_cls"](
            dataset_path=str(cfg["dataset_path"]),
            split=split,
            sampling_rate=SAMPLING_RATE,
        )
    return _dataset_cache[(name, split)]


for name, cfg in ALL_DATASETS.items():
    cfg["assign"] = {
        seed: read_metadata(
            run_dir(cfg, seed) / "learn-prototype-assignments/latest/assignment_metadata.csv"
        )
        for seed in SEEDS
    }
    cfg["proj"] = read_metadata(
        run_dir(cfg, SEED) / "project-prototypes-supervised/latest/projection_metadata.csv"
    )
    cfg["ft_proj"] = read_metadata(
        run_dir(cfg, SEED, f"{MODEL}-ft")
        / "project-prototypes-supervised/latest/projection_metadata.csv"
    )

In [ ]:
# the row position of a prototype in `proj`/`ft_proj` is its slot in the projected
# model, and assignment rows are written label-major then slot-minor, so an
# assignment row's position is exactly that slot. the checks below pin down that
# correspondence, which is what lets us index projections by assignment row.
assert (heedb_proj["prototype_id"] == heedb_proj.index).all()

for name, cfg in ALL_DATASETS.items():
    n_slots = len(cfg["targets"]) * PPL
    for seed, assign in cfg["assign"].items():
        assert len(assign) == n_slots, (name, seed)
        assert (assign.index == assign["label_idx"] * PPL + assign["slot_idx"]).all(), (name, seed)
        # ilp_effect_size uses a prototype at most once per dataset
        assert not assign["prototype_idx"].duplicated().any(), (name, seed)
        # prototypes come from the pretrained bank, which HEEDB projection enumerates
        assert assign["prototype_idx"].isin(heedb_proj.index).all(), (name, seed)
    assert len(cfg["proj"]) == len(cfg["ft_proj"]) == n_slots, name
    assert (cfg["proj"]["prototype_id"] == cfg["proj"].index).all(), name
    assert (cfg["ft_proj"]["prototype_id"] == cfg["ft_proj"].index).all(), name
    # the ILP writes labels in the dataset's target order
    labels_in_order = cfg["assign"][SEED].drop_duplicates("label_idx")["label_name"]
    assert list(labels_in_order) == list(cfg["targets"]), name

# by chance, two random sets of PPL prototypes from the bank share PPL^2 / N
CHANCE_OVERLAP = PPL**2 / N_PROTOTYPES
print(f"{N_PROTOTYPES} pretrained prototypes, chance overlap {CHANCE_OVERLAP:.2f} of {PPL}")
pd.Series(
    {name: cfg["assign"][SEED]["prototype_idx"].nunique() for name, cfg in ALL_DATASETS.items()},
    name=f"prototypes used (seed {SEED})",
).to_frame()

## Screen every EchoNext × MIMIC label pair

Overlap between unrelated labels is not zero: some prototypes seem to mark a generally
abnormal ECG and get picked for many labels. So the right null is not
`CHANCE_OVERLAP` but the overlap of all the other label pairs. Each pair's overlap is
averaged over `SEEDS`.

In [ ]:
def label_prototypes(name: str, target: str, seed: int = SEED) -> set[int]:
    """Pretrained prototype indices dataset `name` assigned to `target` at `seed`."""
    assign = ALL_DATASETS[name]["assign"][seed]
    rows = assign.loc[assign["label_name"] == target]
    assert len(rows) == PPL, f"{name} {target}: {len(rows)} slots != {PPL}"
    return set(rows["prototype_idx"])


def overlap_screen(a: str = "EchoNext", b: str = "MIMIC") -> pd.DataFrame:
    """Shared prototype count for every label pair of datasets `a` and `b`, per seed."""
    rows = []
    for seed in SEEDS:
        sets_a = {t: label_prototypes(a, t, seed) for t in ALL_DATASETS[a]["targets"]}
        sets_b = {t: label_prototypes(b, t, seed) for t in ALL_DATASETS[b]["targets"]}
        for (ta, pa), (tb, pb) in itertools.product(sets_a.items(), sets_b.items()):
            rows.append({a: ta, b: tb, "seed": seed, "shared": len(pa & pb)})
    screen = pd.DataFrame(rows).pivot_table(index=[a, b], columns="seed", values="shared")
    screen["mean"] = screen[SEEDS].mean(axis=1)
    screen["rank"] = screen["mean"].rank(ascending=False, method="min").astype(int)
    screen["percentile"] = screen["mean"].rank(pct=True) * 100
    return screen.sort_values("mean", ascending=False)


screen = overlap_screen()
print(
    f"{len(screen)} pairs, mean shared prototypes: median {screen['mean'].median():.1f}, "
    f"90th pct {screen['mean'].quantile(0.9):.1f}, 99th pct {screen['mean'].quantile(0.99):.1f}"
)
screen.head(10)

In [ ]:
# where our label pairs fall in the screen
screen.loc[[(m["EchoNext"], m["MIMIC"]) for m in LABELS.values()]].set_axis(
    list(LABELS), axis=0
)

In [ ]:
# for scale: the same overlap for a classical label, RBBB in PTB-XL and CODE-15%
pd.Series(
    {
        seed: len(label_prototypes("PTB-XL", "CRBBB", seed) & label_prototypes("CODE-15%", "RBBB", seed))
        for seed in SEEDS
    },
    name="RBBB shared, PTB-XL & CODE-15%",
).to_frame().T.assign(mean=lambda df: df.mean(axis=1))

## Stability across seeds

Seeds change the downstream training (subsets, ILP inputs), not the prototype bank.
A prototype that both datasets share at every seed is a property of the bank, not of
one run.

In [ ]:
def shared_prototypes(label: str, seed: int = SEED) -> set[int]:
    """Prototypes every dataset in `DATASETS` assigned to `label` at `seed`."""
    return set.intersection(
        *(label_prototypes(name, LABELS[label][name], seed) for name in DATASETS)
    )


# prototype -> number of seeds at which it is shared, per label
SEEDS_SHARED = {
    label: collections.Counter(p for seed in SEEDS for p in shared_prototypes(label, seed))
    for label in LABELS
}

display(
    pd.DataFrame(
        {label: {seed: len(shared_prototypes(label, seed)) for seed in SEEDS} for label in LABELS}
    ).rename_axis("seed")
)
pd.DataFrame(
    {
        label: pd.Series(counts).value_counts().sort_index(ascending=False)
        for label, counts in SEEDS_SHARED.items()
    }
).fillna(0).astype(int).rename_axis(f"shared at n of {len(SEEDS)} seeds")

## Overlap at `SEED`

The same Venn regions as in `proto-recur-multilabel.ipynb`, now with two datasets.

In [ ]:
def region_table(sets: dict[str, set[int]]) -> pd.DataFrame:
    """Break `sets` into the mutually exclusive regions of a Venn diagram.

    Every region is listed, including empty ones, so that tables for different
    labels line up. Regions are ordered by how many datasets share them.
    """
    names = list(sets)
    rows = []
    for size in range(1, len(names) + 1):
        for combo in itertools.combinations(names, size):
            shared = set.intersection(*(sets[n] for n in combo))
            others = [sets[n] for n in names if n not in combo]
            members = shared.difference(*others) if others else shared
            rows.append(
                {
                    "region": " & ".join(combo) + " only",
                    "n_datasets": size,
                    "count": len(members),
                    "prototypes": sorted(members),
                }
            )
    out = pd.DataFrame(rows)
    assert out["count"].sum() == len(set().union(*sets.values()))
    return out


PROTOTYPES = {
    label: {name: label_prototypes(name, LABELS[label][name]) for name in DATASETS}
    for label in LABELS
}
pd.concat({label: region_table(sets) for label, sets in PROTOTYPES.items()})

In [ ]:
from matplotlib.patches import Patch
from matplotlib_venn import venn2
from matplotlib_venn.layout.venn2 import DefaultLayoutAlgorithm

assert len(DATASETS) == 2, "venn2 draws exactly two sets"

DATASET_COLORS = dict(zip(DATASETS, ["#2a78d6", "#eb6834"]))  # colorblind-safe

with plt.rc_context({"font.size": 7, "axes.titlesize": 8, "axes.titleweight": "bold"}):
    fig, axs = plt.subplots(1, len(PROTOTYPES), figsize=(TEXT_WIDTH * 2 / 3, 1.75 + 0.3))
    for ax, (label, sets) in zip(np.atleast_1d(axs), PROTOTYPES.items()):
        # equal region sizes give every panel the same geometry
        v = venn2(
            [sets[name] for name in DATASETS],
            set_labels=None,
            set_colors=tuple(DATASET_COLORS.values()),
            layout_algorithm=DefaultLayoutAlgorithm(fixed_subset_sizes=(1,) * 3),
            ax=ax,
        )
        ax.set_title(label)
    # take the swatches from the drawn patches so they match the translucent fill
    handles = [
        Patch(facecolor=v.get_patch_by_id(region).get_facecolor(), label=name)
        for region, name in zip(("10", "01"), DATASETS)
    ]
    fig.legend(
        handles=handles,
        loc="lower center",
        ncol=len(DATASETS),
        frameon=False,
        bbox_to_anchor=(0.5, 0),
    )
    fig.tight_layout(rect=(0, 0.08, 1, 1))
    fig.savefig(FIG_DIR / f"proto-recur-nonclassical-venn-seed{SEED}.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIG_DIR / f"proto-recur-nonclassical-venn-seed{SEED}.pdf", bbox_inches="tight")

## What the shared prototypes mean in classical datasets

Each classical dataset also assigns each prototype to at most one of its own labels,
so a shared prototype can be read through the label PTB-XL, CinC or CODE-15% gave it.
PTB-XL has 71 labels and uses almost the whole bank, so nearly every prototype has a
PTB-XL reading. That means some readings are leftovers the ILP had to place
somewhere. `association_score` (higher is stronger) shows how well each assignment is
supported, and CinC and CODE-15% assign fewer prototypes, so a reading there is
more selective.

`seeds_shared` counts the seeds at which both datasets share the prototype.

In [ ]:
SCP = pd.read_csv(INTERPRETERS["PTB-XL"]["dataset_path"] / "scp_statements.csv", index_col=0)


def ptbxl_class(code: str) -> str:
    """PTB-XL superclass of a statement, with `form`/`rhythm` for non-diagnostic ones.

    `LPR` (prolonged PR) is a form statement in PTB-XL, so it lands under `form`
    even though it is a conduction finding.
    """
    row = SCP.loc[code]
    if row["diagnostic"] == 1:
        return row["diagnostic_class"]
    return "rhythm" if row["rhythm"] == 1 else "form"


def interpret(label: str, seed: int = SEED) -> pd.DataFrame:
    """Every shared prototype of `label`, with its label in each classical dataset."""
    rows = []
    for proto in sorted(shared_prototypes(label, seed)):
        row = {"prototype": proto, "seeds_shared": SEEDS_SHARED[label][proto]}
        for name in DATASETS:
            assign = ALL_DATASETS[name]["assign"][seed].set_index("prototype_idx")
            row[f"{name} score"] = assign.loc[proto, "association_score"]
        for name, cfg in INTERPRETERS.items():
            assign = cfg["assign"][seed].set_index("prototype_idx")
            if proto in assign.index:
                row[name] = assign.loc[proto, "label_name"]
                row[f"{name} score"] = assign.loc[proto, "association_score"]
        if "PTB-XL" in row:
            row["PTB-XL description"] = SCP.loc[row["PTB-XL"], "description"]
            row["PTB-XL class"] = ptbxl_class(row["PTB-XL"])
        rows.append(row)
    return pd.DataFrame(rows).set_index("prototype")


INTERPRETATIONS = {label: interpret(label) for label in LABELS}
for label, table in INTERPRETATIONS.items():
    print(label)
    display(table.round(2).fillna("-"))

### Enrichment by PTB-XL class

The PTB-XL class of the shared prototypes, compared with the prototypes only one of
the two datasets picked for the label, and with the whole bank. Counts are pooled
over `SEEDS`. The same prototype recurs across seeds, so the pooled counts are not
independent and the table is descriptive, not a test.

In [ ]:
def class_enrichment(label: str) -> pd.DataFrame:
    """Share of each PTB-XL class among shared, one-dataset-only and all prototypes."""
    groups = collections.defaultdict(list)
    for seed in SEEDS:
        ptbxl = INTERPRETERS["PTB-XL"]["assign"][seed].set_index("prototype_idx")["label_name"]
        classes = ptbxl.map(ptbxl_class).reindex(heedb_proj.index).fillna("unassigned")
        sets = [label_prototypes(name, LABELS[label][name], seed) for name in DATASETS]
        shared = set.intersection(*sets)
        groups["shared"] += list(classes[sorted(shared)])
        groups["one dataset only"] += list(classes[sorted(set.union(*sets) - shared)])
        groups["all prototypes"] += list(classes)
    out = pd.DataFrame(
        {group: pd.Series(values).value_counts(normalize=True) for group, values in groups.items()}
    ).fillna(0)
    out.loc["n (pooled)"] = [len(values) for values in groups.values()]
    return out.sort_values("shared", ascending=False)


for label in LABELS:
    print(label)
    display(class_enrichment(label).round(3))

## Case studies: projected windows only

As in `proto-recur-multilabel.ipynb`, each run projects a prototype onto the training
sample and 1-second window that activate it most, so each run has its own source
waveform. Rows are runs, grouped by dataset; columns are the leads chosen per case.
The classical dataset rows show the projection for the label that dataset assigned
the prototype to (in the group name), which is the ECG reading of the prototype.

`has_label` checks that each source ECG carries the label that run projected for. The
HEEDB row (`INCLUDE_PRETRAIN`) is a label-free projection, so it has no label to check.

The examples are prototypes with clear classical readings, shared at every seed: 217
and 991 for LV systolic dysfunction, 728 and 992 for aortic valve disease. 991 turns
out to be one of the two candidates in the next section.

In [ ]:
RUN_NAMES = {"pretrain": "pretrained", "probe": "linear probe", "ft": "fine-tuned"}
CASE_INTERPRETERS = ["PTB-XL"]  # classical datasets to show, each adds two rows


def prototype_sources(
    proto_idx: int,
    interpreters: list[str] = CASE_INTERPRETERS,
    include_pretrain: bool = INCLUDE_PRETRAIN,
) -> pd.DataFrame:
    """Sample and window each run projected prototype `proto_idx` onto, at `SEED`.

    One row per run: the HEEDB pretraining projection (optional), then the linear
    probe and fine-tuned projections of every dataset (and interpreter) that
    assigned the prototype, with the label it was assigned to.
    """
    rows = []
    if include_pretrain:
        rows.append(
            {"dataset": "HEEDB", "run": "pretrain", "label": None, **heedb_proj.loc[proto_idx]}
        )
    for name in [*DATASETS, *interpreters]:
        cfg = ALL_DATASETS[name]
        assign = cfg["assign"][SEED]
        slots = assign.index[assign["prototype_idx"] == proto_idx]
        if len(slots) == 0:
            continue  # this dataset did not use the prototype for any label
        slot = slots[0]  # at most one, checked when loading
        target = assign.loc[slot, "label_name"]
        rows.append({"dataset": name, "run": "probe", "label": target, **cfg["proj"].loc[slot]})
        rows.append({"dataset": name, "run": "ft", "label": target, **cfg["ft_proj"].loc[slot]})
    columns = ["dataset", "run", "label", "source_id", "sample_id", "chunk_idx", "emb_sim"]
    ids = ["source_id", "sample_id", "chunk_idx"]  # floated by the mixed-dtype rows
    return pd.DataFrame(rows)[columns].astype(dict.fromkeys(ids, int))


def prototype_windows(
    proto_idx: int, split: str = SPLIT, **kwargs
) -> tuple[pd.DataFrame, list[np.ndarray]]:
    """Projected 1-second (L, T) window of `proto_idx` in every run that uses it.

    Also flags whether each window's source ECG carries the label of its run.
    """
    sources = prototype_sources(proto_idx, **kwargs)
    windows, has_label = [], []
    for _, row in sources.iterrows():
        dataset = get_dataset(row["dataset"], split=split)
        idx = resolve_index(dataset, sample_id=int(row["sample_id"]))
        signal = get_ecg_waveform(
            dataset=dataset, index=idx, sampling_rate=SAMPLING_RATE
        )
        # same windowing as plot_ecg's chunk_idx highlight
        start, end = chunk_to_seconds(
            int(row["chunk_idx"]),
            chunk_len=SAMPLING_RATE,
            chunk_overlap=0.5,
            sampling_rate=SAMPLING_RATE,
        )
        windows.append(signal[:, int(start * SAMPLING_RATE) : int(end * SAMPLING_RATE)])
        if row["label"] is None:
            has_label.append(None)  # label-free pretraining projection
        else:
            targets = list(_datasets[row["dataset"]]["targets"])
            has_label.append(bool(dataset[idx]["label"][targets.index(row["label"])]))
    return sources.assign(has_label=has_label), windows

In [ ]:
def plot_prototype_windows(
    sources: pd.DataFrame,
    windows: list[np.ndarray],
    leads: list[str],
    row_height: float = 0.8,
) -> plt.Figure:
    """Grid of projected windows, one row per run and one column per lead.

    Classical datasets are named with the label they assigned the prototype to.
    """
    group_names = {
        dataset: f"{dataset}: {label}" if dataset in INTERPRETERS else dataset
        for dataset, label in zip(sources["dataset"], sources["label"])
    }
    # one grid row per run, plus a thin spacer row wherever the dataset changes
    height_ratios, grid_rows = [], []
    for i, dataset in enumerate(sources["dataset"]):
        if i > 0 and dataset != sources["dataset"].iloc[i - 1]:
            height_ratios.append(0.25)
        grid_rows.append(len(height_ratios))
        height_ratios.append(1)

    with plt.rc_context({"font.size": 7}):
        fig = plt.figure(figsize=(TEXT_WIDTH, row_height * sum(height_ratios) + 0.2))
        gs = fig.add_gridspec(
            len(height_ratios),
            len(leads),
            height_ratios=height_ratios,
            hspace=0.12,
            wspace=0.06,
        )
        group_axs: dict[str, list[plt.Axes]] = {}
        for (_, row), grid_row, window in zip(sources.iterrows(), grid_rows, windows):
            axs = np.array([[fig.add_subplot(gs[grid_row, j]) for j in range(len(leads))]])
            plot_segment(
                window,
                axs,
                sampling_rate=SAMPLING_RATE,
                lead_layout=[leads],
                label_leads=False,  # labeled once as column headers instead
            )
            axs[0, 0].set_ylabel(
                RUN_NAMES[row["run"]], rotation=0, ha="right", va="center", labelpad=4
            )
            group_axs.setdefault(row["dataset"], []).append(axs[0, 0])
            if grid_row == 0:
                for ax, lead in zip(axs.flat, leads):
                    ax.set_title(lead, fontweight="bold", fontsize=8)

        # dataset names go left of the run names, centered on each group's rows
        fig.canvas.draw()
        to_fig = fig.transFigure.inverted()
        run_label_x0 = min(
            ax.yaxis.label.get_window_extent().x0
            for axs in group_axs.values()
            for ax in axs
        )
        x = to_fig.transform((run_label_x0, 0))[0] - 0.015
        for dataset, axs in group_axs.items():
            y = (axs[0].get_position().y1 + axs[-1].get_position().y0) / 2
            fig.text(
                x, y, group_names[dataset], rotation=90, ha="right", va="center",
                fontweight="bold", fontsize=8,
            )
    return fig

In [ ]:
# (concept, prototype, leads): leads chosen for the prototype's classical reading
CASE_STUDIES = [
    # PTB-XL: ST-T changes of ventricular aneurysm, i.e. old anterior MI
    ("LV systolic dysfunction", 217, ["V1", "V2", "V3", "V4"]),
    # PTB-XL: incomplete LBBB, CODE-15%: LBBB
    ("LV systolic dysfunction", 991, ["I", "V1", "V5", "V6"]),
    # PTB-XL: LVH
    ("Aortic valve disease", 728, ["aVL", "V1", "V5", "V6"]),
    # PTB-XL: prolonged PR, CinC: 1st-degree AV block, CODE-15%: 1st-degree AV block
    ("Aortic valve disease", 992, ["I", "II", "V1", "V5"]),
]

for label, proto_idx, leads in CASE_STUDIES:
    assert proto_idx in shared_prototypes(label), f"{proto_idx} not shared for {label}"
    sources, windows = prototype_windows(proto_idx)
    print(f"{label} | prototype {proto_idx}")
    display(sources)
    assert sources["has_label"].dropna().all()

    fig = plot_prototype_windows(sources, windows, leads)
    stem = f"{LABEL_SLUGS[label]}-p{proto_idx}-windows-seed{SEED}"
    fig.savefig(FIG_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{stem}.pdf", bbox_inches="tight")
    plt.show()

## Beyond known ECG findings

ECG is not how these diseases are diagnosed, and how they show on an ECG is only
partly understood. Everything above shows that the shared prototypes line up with known
correlates (old MI, LBBB, LVH, conduction disease). The question here is whether any
of them carry signal those known findings **don't** explain, which is what would make
it a candidate for something new to read off the ECG.

**Design (provisional, to be revised with a cardiologist).** We use the per-ECG
prototype similarities the runs saved (`compute-embeddings`, one column per assigned
slot, at `SEED`).

1. **Incremental value.** A known-findings model (logistic regression on the
   covariates below, continuous ones as cubic splines, see `INTERVAL_TERMS`) is fit
   on train. On test we ask (a) whether adding the shared
   prototypes raises AUROC, with a bootstrap CI, and (b) per prototype, its odds ratio
   per SD adjusted for the known-findings score (the base model's logit), with
   patient-clustered SEs. `attenuation` is the share of the unadjusted log-odds the
   known findings explain.
2. **ECGs with no known findings.** In the normal-read subset of test, the
   prototype's odds ratio per SD adjusted for age (quadratic) and sex, and its AUROC.

Inference is on **test** because the ILP chose the prototypes using train labels, so
effects estimated on train would be inflated.

**Known findings.**

- Both datasets: age, sex, heart rate, PR, QRS duration, QTc. EchoNext adds the atrial
  rate; MIMIC adds the QRS and T axes, derived from `machine_measurements.csv`.
- MIMIC only: machine-read statements (`KNOWN_STATEMENTS`, matched by regex):
  infarct/Q waves, LBBB, incomplete LBBB, RBBB, IVCD, fascicular block, LVH, left atrial
  abnormality, low voltage, R-wave progression, ST-T abnormality, AF/flutter, paced,
  ectopy, AV block. EchoNext ships no statements, so on its own its adjustment is
  intervals only, a much weaker control.
- Both datasets, added in the imputed variants: PTB-XL's 71 statements imputed on
  every ECG through PTB-XL's PILA model and linear probe
  (`scripts/8-run-protossl-heedb-pila-from-ptbxl.sh`), checked against MIMIC's machine
  reads below.

Every analysis runs under each variant in `KNOWN_VARIANTS`:

- **as recorded**: intervals, plus statements in MIMIC.
- **+ imputed reads** (the main one): as recorded plus each imputed finding as present
  or absent, the way a reader would record it.
- **+ imputed probabilities**: as recorded plus all 71 imputed log-odds. This one
  over-adjusts (see the caveat below) and is kept to show by how much.

**Normal-read.**

- MIMIC: no known statement, QRS < 120 ms, PR < 200 ms, heart rate 50–100, QRS axis
  −30 to 90°.
- EchoNext: the same interval criteria, plus atrial rate within 5 bpm of the
  ventricular rate as a stand-in for sinus rhythm with 1:1 conduction.
- Both imputed variants additionally require no reliably imputed finding to be
  present (see the validation below).

**Candidate criteria.**

- *Beyond known findings*: adjusted OR > 1 with Holm-corrected p < 0.05 (over all
  prototypes, labels and datasets of a variant) in **both** datasets.
- *In normal-read ECGs*: age/sex-adjusted OR > 1 with nominal p < 0.05 in every
  dataset whose subset has at least `MIN_NORMAL_POS` positives (in practice EchoNext
  only, see below).

In [ ]:
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer, StandardScaler
from statsmodels.stats.multitest import multipletests

# machine-read statements counted as known findings, matched case-insensitively
# against all of an ECG's report_* lines
KNOWN_STATEMENTS = {
    "infarct / Q waves": r"infarct|q waves?|\bmi\b",
    "LBBB": r"(?<!incomplete )left bundle branch block|\blbbb\b",
    "incomplete LBBB": r"incomplete left bundle branch block",
    "RBBB": r"right bundle branch block|\brbbb\b",
    "IVCD": r"intraventricular conduction|iv conduction|\bivcd\b",
    "fascicular block": r"fascicular block|\blafb\b|\blpfb\b|hemiblock",
    "LVH": r"left ventricular hypertrophy|\blvh\b",
    "left atrial abnormality": r"left atrial (?:enlargement|abnormality)",
    "low voltage": r"low (?:qrs )?voltage",
    "R-wave progression": r"r[- ]wave progression",
    "ST-T abnormality": r"\bst\b|st-t|t wave|\bt abn|repol|ischemi|injury",
    "AF / flutter": r"atrial fibrillation|atrial flutter",
    "paced": r"pacemaker|paced|pacing",
    "ectopy": r"premature|\bpacs?\b|\bpvcs?\b|ectopic|bigeminy|trigeminy",
    "AV block": r"a-?v block|prolonged pr",
}
MIN_NORMAL_POS = 20  # fewer positives than this in the normal-read subset: not evaluated

MACHINE_MEASUREMENTS = pd.read_csv(
    DATASETS["MIMIC"]["dataset_path"] / "machine_measurements.csv", low_memory=False
).set_index("study_id")


def in_range(x: pd.Series, lo: float, hi: float) -> pd.Series:
    """`x` with values outside [lo, hi] (sentinels like 29999) set to NaN."""
    x = x.astype(float)
    return x.where((x >= lo) & (x <= hi))


def mimic_covariates(df: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray]:
    """Known findings for MIMIC rows `df`, and which rows are normal-read."""
    mm = MACHINE_MEASUREMENTS.loc[df["study_id"]].reset_index()
    # fiducial points are ms offsets into the recording
    rr = in_range(mm["rr_interval"], 250, 3000)
    cov = pd.DataFrame(
        {
            "age": df["age"].to_numpy(),
            "sex": df["gender"].to_numpy(),
            "heart rate": 60000 / rr,
            "PR": in_range(mm["qrs_onset"] - mm["p_onset"], 60, 400),
            "QRS": in_range(mm["qrs_end"] - mm["qrs_onset"], 40, 250),
            "QTc": in_range(mm["t_end"] - mm["qrs_onset"], 200, 700) / np.sqrt(rr / 1000),
            "QRS axis": in_range(mm["qrs_axis"], -180, 180),
            "T axis": in_range(mm["t_axis"], -180, 180),
        }
    )
    report_cols = [c for c in mm.columns if c.startswith("report_")]
    text = mm[report_cols].fillna("").agg(" | ".join, axis=1).str.lower()
    for finding, pattern in KNOWN_STATEMENTS.items():
        cov[finding] = text.str.contains(pattern, regex=True).astype(int).to_numpy()
    normal = (
        ~cov[list(KNOWN_STATEMENTS)].any(axis=1)
        & (cov["QRS"] < 120)
        & (cov["PR"] < 200)
        & cov["heart rate"].between(50, 100)
        & cov["QRS axis"].between(-30, 90)
    )
    return cov, normal.to_numpy()


def echonext_covariates(df: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray]:
    """Known findings for EchoNext rows `df` (intervals only), and normal-read rows."""
    cov = pd.DataFrame(
        {
            "age": df["age_at_ecg"].to_numpy(),
            "sex": (df["sex"] == "male").astype(int).to_numpy(),
            "heart rate": in_range(df["ventricular_rate"], 20, 250),
            "atrial rate": in_range(df["atrial_rate"], 20, 600),
            "PR": in_range(df["pr_interval"], 60, 400),
            "QRS": in_range(df["qrs_duration"], 40, 250),
            "QTc": in_range(df["qt_corrected"], 200, 700),
        }
    )
    normal = (
        (cov["QRS"] < 120)
        & (cov["PR"] < 200)
        & cov["heart rate"].between(50, 100)
        & ((cov["atrial rate"] - cov["heart rate"]).abs() <= 5)
    )
    return cov, normal.to_numpy()


COVARIATES = {"EchoNext": echonext_covariates, "MIMIC": mimic_covariates}

### Imputed PTB-XL statements

`scripts/8-run-protossl-heedb-pila-from-ptbxl.sh` ran every EchoNext and MIMIC ECG
through PTB-XL's `protossl-heedb-pila` model (at `SEED`) and its linear probe, giving a
probability for each of PTB-XL's 71 statements. MIMIC has machine reads to check them
against. `IMPUTED_FINDINGS` maps each known finding to the PTB-XL codes that express it
(a finding's score is the maximum over its codes).

An imputed finding counts as **present** when its score reaches a cutoff chosen to
flag the same share of MIMIC **train** ECGs as the machine reads do. A fixed 0.5
cutoff wouldn't work: the probe's probabilities aren't calibrated to MIMIC (left
atrial abnormality averages 2% imputed vs 12% machine-read). The same cutoffs carry
over to EchoNext. Only findings the probe recovers with AUROC ≥ `MIN_IMPUTED_AUROC` on
MIMIC train count toward normal-read; the rest are too noisy to exclude ECGs on.

**Caveat: over-adjustment.** Each imputed log-odds is a linear function of PTB-XL's
994 prototype similarities. On EchoNext and MIMIC those rebuild about 90% of any
prototype's similarity, with or without the tested prototype's own slot (see the
exploratory section). **+ imputed probabilities** therefore partly adjusts for the
tested prototype itself and pulls every prototype's effect toward zero, candidates
and controls alike, so failing under it says nothing about a particular prototype.
Present/absent reads (**+ imputed reads**) keep only what a reader would record,
which is the fair comparison. The exploratory section adds graded known findings that
don't share ProtoSSL's features.

In [ ]:
import json

SHARED_ROOT = Path(os.environ.get("PROTOSSL_SHARED_ROOT", "/shared/gpudata/songs1/protossl-shared"))
IMPUTED_RUN = "recur"  # run that published protossl-heedb-pila-from-ptbxl
IMPUTED_EXP = "protossl-heedb-pila-from-ptbxl"
MIN_IMPUTED_AUROC = 0.8


def imputed_dir(name: str, seed: int = SEED) -> Path:
    slug = ALL_DATASETS[name]["slug"]
    return SHARED_ROOT / "runs" / IMPUTED_RUN / f"protossl-outputs-seed{seed}" / f"runs-{slug}" / IMPUTED_EXP


IMPUTED_LABELS = json.loads((imputed_dir("MIMIC") / "label_names.json").read_text())
assert IMPUTED_LABELS == list(PTBXL_TARGETS)
for name in DATASETS:
    assert json.loads((imputed_dir(name) / "label_names.json").read_text()) == IMPUTED_LABELS
    source = (imputed_dir(name) / "SOURCE").read_text().strip()
    assert source.endswith(f"protossl-outputs-seed{SEED}/runs-ptbxl/{MODEL}"), source

_ptbxl_codes = lambda superclass: [c for c in IMPUTED_LABELS if ptbxl_class(c) == superclass]
# known finding -> PTB-XL codes that express it (no PTB-XL code for R-wave progression)
IMPUTED_FINDINGS = {
    "infarct / Q waves": _ptbxl_codes("MI") + ["QWAVE"],
    "LBBB": ["CLBBB"],
    "incomplete LBBB": ["ILBBB"],
    "RBBB": ["CRBBB", "IRBBB"],
    "IVCD": ["IVCD"],
    "fascicular block": ["LAFB", "LPFB"],
    "LVH": ["LVH", "VCLVH"],
    "left atrial abnormality": ["LAO/LAE"],
    "low voltage": ["LVOLT"],
    "ST-T abnormality": _ptbxl_codes("STTC") + ["STD_", "STE_", "INVT", "LOWT", "NT_", "TAB_"],
    "AF / flutter": ["AFIB", "AFLT"],
    "paced": ["PACE"],
    "ectopy": ["PVC", "PAC", "PRC(S)", "BIGU", "TRIGU"],
    "AV block": ["1AVB", "2AVB", "3AVB", "LPR"],
}
assert set(IMPUTED_FINDINGS) <= set(KNOWN_STATEMENTS)


def load_imputed(
    name: str, split: str, directory: Path | None = None
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Imputed statement log-odds (one column per PTB-XL code), and finding scores."""
    probs = np.load((directory or imputed_dir(name)) / f"{split}_probs.npy")
    assert probs.shape[1] == len(IMPUTED_LABELS)
    codes = pd.DataFrame(probs, columns=IMPUTED_LABELS)
    scores = pd.DataFrame(
        {finding: codes[cols].max(axis=1) for finding, cols in IMPUTED_FINDINGS.items()}
    )
    p = np.clip(probs, 1e-6, 1 - 1e-6)
    log_odds = pd.DataFrame(np.log(p / (1 - p)), columns=[f"PTB-XL {c}" for c in IMPUTED_LABELS])
    return log_odds, scores

In [ ]:
def load_split(
    name: str, split: str, embeds_dir: Path | None = None, imputed_from: Path | None = None
) -> dict:
    """Prototype similarities, labels, known findings and patient IDs of one split.

    Similarities and imputed statements default to the `SEED` run's outputs; the val
    split's come from the replication job instead (see the end of this section).
    """
    dataset = get_dataset(name, split)
    cfg = ALL_DATASETS[name]
    embeds_dir = embeds_dir or run_dir(cfg, SEED) / "compute-embeddings/latest"
    embeds = np.load(embeds_dir / f"{split}_embeds.npy")
    # which window each prototype matched; only newer runs (e.g. the val job) save it
    chunks_file = embeds_dir / f"{split}_chunks.npy"
    chunks = np.load(chunks_file) if chunks_file.exists() else None
    labels = dataset.labels.numpy()
    assert embeds.shape == (len(labels), len(cfg["targets"]) * PPL), (name, split)
    cov, normal = COVARIATES[name](dataset._df)
    imputed, imputed_scores = load_imputed(name, split, imputed_from)
    assert len(imputed) == len(labels), (name, split)
    return {
        "embeds": embeds,
        "chunks": chunks,
        "labels": labels,
        "cov": cov,
        "normal": normal,
        "imputed": imputed,
        "imputed_scores": imputed_scores,
        "patients": dataset.source_ids.numpy(),
    }


SPLITS = {(name, split): load_split(name, split) for name in DATASETS for split in ["train", "test"]}


def check_alignment(split: str) -> None:
    """A label's own slots must predict it better than other slots, which pins down
    that embedding rows follow the dataset and columns follow assignment rows."""
    for name in DATASETS:
        data, assign = SPLITS[name, split], ALL_DATASETS[name]["assign"][SEED]
        for label, mapping in LABELS.items():
            k = list(ALL_DATASETS[name]["targets"]).index(mapping[name])
            own = (assign["label_name"] == mapping[name]).to_numpy()
            y = data["labels"][:, k]
            auc = np.array([roc_auc_score(y, data["embeds"][:, j]) for j in range(len(assign))])
            assert auc[own].mean() > auc[~own].mean() + 0.05, (name, split, label)


check_alignment("test")

In [ ]:
_mimic_train = SPLITS["MIMIC", "train"]
# finding -> score at which it's present: flags MIMIC train's machine-read share
IMPUTED_CUTOFFS = {
    finding: np.quantile(
        _mimic_train["imputed_scores"][finding], 1 - _mimic_train["cov"][finding].mean()
    )
    for finding in IMPUTED_FINDINGS
}


def imputed_reads(data: dict) -> pd.DataFrame:
    """Each imputed finding as present (1) or absent (0)."""
    return pd.DataFrame(
        {
            f"imputed {finding}": (data["imputed_scores"][finding] >= cutoff).astype(int)
            for finding, cutoff in IMPUTED_CUTOFFS.items()
        }
    )


def validate_imputed(split: str) -> pd.DataFrame:
    """How well each imputed finding recovers MIMIC's machine read."""
    data = SPLITS["MIMIC", split]
    reads = imputed_reads(data)
    rows = []
    for finding in IMPUTED_FINDINGS:
        machine = data["cov"][finding].to_numpy().astype(bool)
        score = data["imputed_scores"][finding].to_numpy()
        imputed = reads[f"imputed {finding}"].to_numpy().astype(bool)
        rows.append(
            {
                "finding": finding,
                "machine-read prevalence": machine.mean(),
                "mean imputed probability": score.mean(),
                "AUROC": roc_auc_score(machine, score),
                "sensitivity": imputed[machine].mean(),
                "specificity": (~imputed[~machine]).mean(),
            }
        )
    return pd.DataFrame(rows).set_index("finding")


IMPUTED_VALIDATION = validate_imputed("train")
RELIABLE_FINDINGS = list(
    IMPUTED_VALIDATION.index[IMPUTED_VALIDATION["AUROC"] >= MIN_IMPUTED_AUROC]
)
display(
    pd.concat(
        {"train": IMPUTED_VALIDATION, "test": validate_imputed("test")}, axis=1
    ).round(3)
)
print("used for normal-read:", RELIABLE_FINDINGS)

In [ ]:
def imputed_normal(data: dict) -> np.ndarray:
    """No reliably imputed finding present."""
    reads = imputed_reads(data)[[f"imputed {f}" for f in RELIABLE_FINDINGS]]
    return (reads.sum(axis=1) == 0).to_numpy()


# known-findings variant -> (covariates, normal-read mask) of one split
KNOWN_VARIANTS = {
    "as recorded": lambda data: (data["cov"], data["normal"]),
    "+ imputed reads": lambda data: (
        pd.concat([data["cov"], imputed_reads(data)], axis=1),
        data["normal"] & imputed_normal(data),
    ),
    "+ imputed probabilities": lambda data: (
        pd.concat([data["cov"], data["imputed"]], axis=1),
        data["normal"] & imputed_normal(data),
    ),
}

counts = []
for variant, get_known in KNOWN_VARIANTS.items():
    for name in DATASETS:
        test = SPLITS[name, "test"]
        _, normal = get_known(test)
        for label, mapping in LABELS.items():
            y = test["labels"][:, list(ALL_DATASETS[name]["targets"]).index(mapping[name])]
            counts.append(
                {
                    "known findings": variant,
                    "label": label,
                    "dataset": name,
                    "test ECGs": len(y),
                    "positives": int(y.sum()),
                    "normal-read ECGs": int(normal.sum()),
                    "normal-read positives": int(y[normal].sum()),
                }
            )
pd.DataFrame(counts).set_index(["known findings", "label", "dataset"])

In [ ]:
def design_matrix(cov: pd.DataFrame, train_cov: pd.DataFrame) -> pd.DataFrame:
    """Median-impute with train medians, adding a missing indicator where needed."""
    X = cov.copy()
    for col in train_cov.columns:
        if train_cov[col].isna().any():
            X[f"{col} missing"] = X[col].isna().astype(int)
        X[col] = X[col].fillna(train_cov[col].median())
    return X


# continuous known findings, entered as splines unless INTERVAL_TERMS == "linear": the
# risk of low EF, for one, rises steeply once QRS passes ~120 ms, which a straight
# line would leave for a QRS-width prototype to pick up
SPLINE_COVARIATES = ["age", "heart rate", "atrial rate", "PR", "QRS", "QTc", "QRS axis", "T axis"]
INTERVAL_TERMS = "spline"


def expand_terms(
    X_train: pd.DataFrame, X_test: pd.DataFrame, terms: str = INTERVAL_TERMS
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Replace continuous covariates by cubic B-spline bases (knots at train quantiles)."""
    if terms == "linear":
        return X_train, X_test
    assert terms == "spline", terms
    cols = [c for c in SPLINE_COVARIATES if c in X_train]
    # include_bias=False: a full B-spline basis sums to 1, collinear with the intercept
    splines = SplineTransformer(
        n_knots=5, degree=3, knots="quantile", extrapolation="linear", include_bias=False
    )
    splines.fit(X_train[cols])
    names = splines.get_feature_names_out(cols)

    def expand(X):
        basis = pd.DataFrame(splines.transform(X[cols]), columns=names, index=X.index)
        return pd.concat([X.drop(columns=cols), basis], axis=1)

    return expand(X_train), expand(X_test)


def logit(y, X, groups):
    """Logistic regression with patient-clustered standard errors."""
    return sm.Logit(y, sm.add_constant(X)).fit(
        disp=0, cov_type="cluster", cov_kwds={"groups": groups}
    )


def beyond_known(
    variant: str,
    terms: str = INTERVAL_TERMS,
    eval_split: str = "test",
    prototypes: dict[str, list[int]] | None = None,
    variants: dict | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Steps 1 and 2 under one known-findings variant: (AUROC gains, per prototype).

    Models are fit on train and evaluated on `eval_split`. `prototypes` restricts the
    labels and prototypes tested (default: every shared prototype of every label);
    the Holm correction runs over exactly the tests made. `variants` looks `variant`
    up somewhere other than `KNOWN_VARIANTS` (the exploratory section's).
    """
    get_known = (variants or KNOWN_VARIANTS)[variant]
    rng = np.random.default_rng(0)
    if prototypes is None:
        prototypes = {label: sorted(shared_prototypes(label)) for label in LABELS}
    gain_rows, proto_rows = [], []
    for label, protos in prototypes.items():
        mapping = LABELS[label]
        for name in DATASETS:
            gains, rows = _beyond_known_one(
                label, mapping, name, protos, get_known, terms, rng, eval_split
            )
            gain_rows.append({"label": label, "dataset": name, **gains})
            proto_rows += [{"label": label, "dataset": name, **row} for row in rows]
    gains = pd.DataFrame(gain_rows).set_index(["label", "dataset"])
    beyond = pd.DataFrame(proto_rows)
    beyond["p adjusted (Holm)"] = multipletests(beyond["p adjusted"], method="holm")[1]
    return gains, beyond.set_index(["label", "prototype", "dataset"]).sort_index()


def _beyond_known_one(label, mapping, name, protos, get_known, terms, rng, eval_split):
    # `test` is whichever split is evaluated
    train, test = SPLITS[name, "train"], SPLITS[name, eval_split]
    k = list(ALL_DATASETS[name]["targets"]).index(mapping[name])
    y_train, y_test = train["labels"][:, k], test["labels"][:, k]
    cov_train, _ = get_known(train)
    cov_test, normal = get_known(test)
    X_train = design_matrix(cov_train, cov_train)
    X_test = design_matrix(cov_test, cov_train)[X_train.columns]
    X_train, X_test = expand_terms(X_train, X_test, terms)
    # each prototype's column is the slot this dataset assigned it to
    cols = [
        int(ALL_DATASETS[name]["assign"][SEED].index[
            ALL_DATASETS[name]["assign"][SEED]["prototype_idx"] == p
        ][0])
        for p in protos
    ]
    mu, sd = train["embeds"][:, cols].mean(0), train["embeds"][:, cols].std(0)
    Z_train = (train["embeds"][:, cols] - mu) / sd
    Z_test = (test["embeds"][:, cols] - mu) / sd

    # step 1a: known findings vs known findings + shared prototypes, out of sample
    known = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
    known.fit(X_train, y_train)
    both = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
    both.fit(np.c_[X_train, Z_train], y_train)
    p_known = known.predict_proba(X_test)[:, 1]
    p_both = both.predict_proba(np.c_[X_test, Z_test])[:, 1]
    deltas = []
    for _ in range(1000):
        i = rng.integers(0, len(y_test), len(y_test))
        if y_test[i].min() != y_test[i].max():
            deltas.append(roc_auc_score(y_test[i], p_both[i]) - roc_auc_score(y_test[i], p_known[i]))
    gains = (
        {
            "AUROC known": roc_auc_score(y_test, p_known),
            "AUROC known + shared": roc_auc_score(y_test, p_both),
            "ΔAUROC": roc_auc_score(y_test, p_both) - roc_auc_score(y_test, p_known),
            "95% CI lo": np.percentile(deltas, 2.5),
            "95% CI hi": np.percentile(deltas, 97.5),
        }
    )

    # step 1b and step 2, one prototype at a time, on test
    known_score = known.decision_function(X_test)
    rows = []
    n_normal_pos = int(y_test[normal].sum())
    # age as a quadratic, not a spline: the subset is small (as few as ~30 positives)
    # and sparse tail basis functions would separate
    age = (cov_test.loc[normal, "age"].to_numpy(float) - 60) / 15
    age_sex = np.c_[age, age**2, cov_test.loc[normal, "sex"].to_numpy(float)]
    for j, proto in enumerate(protos):
        z = Z_test[:, j]
        unadj = logit(y_test, z, test["patients"])
        adj = logit(y_test, np.c_[known_score, z], test["patients"])
        row = {
            "prototype": proto,
            "OR unadjusted": np.exp(unadj.params[-1]),
            "OR adjusted": np.exp(adj.params[-1]),
            "OR adj lo": np.exp(adj.conf_int()[-1, 0]),
            "OR adj hi": np.exp(adj.conf_int()[-1, 1]),
            "p adjusted": adj.pvalues[-1],
            "attenuation": 1 - adj.params[-1] / unadj.params[-1],
            "normal-read positives": n_normal_pos,
        }
        if n_normal_pos >= MIN_NORMAL_POS:
            nr = logit(y_test[normal], np.c_[age_sex, z[normal]], test["patients"][normal])
            row |= {
                "normal-read AUROC": roc_auc_score(y_test[normal], z[normal]),
                "normal-read OR": np.exp(nr.params[-1]),
                "normal-read p": nr.pvalues[-1],
            }
        rows.append(row)
    return gains, rows


RESULTS = {variant: beyond_known(variant) for variant in KNOWN_VARIANTS}
GAINS = pd.concat({v: gains for v, (gains, _) in RESULTS.items()}, names=["known findings"])
BEYOND = pd.concat({v: beyond for v, (_, beyond) in RESULTS.items()}, names=["known findings"])
GAINS.round(3)

In [ ]:
BEYOND.round(3)

In [ ]:
def candidates() -> pd.DataFrame:
    """Which shared prototypes meet each provisional criterion.

    *Beyond known findings* must hold in both datasets. *In normal-read ECGs* is
    judged over the datasets with at least `MIN_NORMAL_POS` normal-read positives,
    listed in `normal-read datasets` (`None` when there are none).
    """
    wide = BEYOND.unstack("dataset")  # rows: (known findings, label, prototype)
    beyond = ((wide["OR adjusted"] > 1) & (wide["p adjusted (Holm)"] < 0.05)).all(axis=1)
    if "normal-read OR" in wide:
        evaluable = wide["normal-read OR"].notna()
        hit = (wide["normal-read OR"] > 1) & (wide["normal-read p"] < 0.05)
        normal = (hit | ~evaluable).all(axis=1).astype(object).where(evaluable.any(axis=1), None)
        normal_datasets = evaluable.apply(lambda r: ", ".join(r.index[r]) or None, axis=1)
    else:
        normal = pd.Series(None, index=wide.index, dtype=object)
        normal_datasets = normal
    ptbxl = pd.concat(
        {label: INTERPRETATIONS[label]["PTB-XL"] for label in LABELS}, names=["label", "prototype"]
    )
    table = pd.DataFrame(
        {
            "beyond known findings": beyond,
            "in normal-read ECGs": normal,
            "normal-read datasets": normal_datasets,
        }
    )
    table = table.unstack("known findings").swaplevel(axis=1)[list(KNOWN_VARIANTS)]
    table.insert(0, "PTB-XL reading", ptbxl.reindex(table.index))
    return table


candidates()

### Linear vs spline interval terms

The same analysis under the main variant with continuous covariates entered as
straight lines, to show what the splines change. A prototype whose adjusted effect
shrinks a lot under splines was partly encoding a nonlinear interval effect (e.g. QRS
width).

In [ ]:
MAIN_VARIANT = "+ imputed reads"
linear_gains, linear_beyond = beyond_known(MAIN_VARIANT, terms="linear")
spline_gains, spline_beyond = RESULTS[MAIN_VARIANT]
display(
    pd.concat({"linear": linear_gains["ΔAUROC"], "spline": spline_gains["ΔAUROC"]}, axis=1).round(3)
)
compare = ["OR adjusted", "p adjusted (Holm)"]
pd.concat({"linear": linear_beyond[compare], "spline": spline_beyond[compare]}, axis=1).unstack(
    "dataset"
).round(3)

### Reading these results

- **Effects can point either way.** The ILP assigns by effect size, so a prototype
  can be negatively associated with its label (OR < 1). Such prototypes don't count
  as candidates under the criteria above, but they're still informative.
- **EchoNext vs MIMIC.** As recorded, EchoNext's known-findings model misses infarct,
  LBBB, LVH and so on, so a prototype can look "beyond known" there just by encoding
  one of them. That's why the criterion requires MIMIC too, and why the imputed
  variants exist.
- **Over-adjustment** (see the caveat above): + imputed probabilities absorbs
  nearly every prototype, so failing only there isn't evidence against one in
  particular.
- **Normal-read isn't specific.** Under the main variant, the known-pattern controls
  217, 522, 578 and 617 also pass it in EchoNext, so passing it doesn't single out a
  candidate.
- **MIMIC's normal-read subset barely contains these diseases.** A MIMIC ED patient
  with HFrEF or aortic valve disease almost always has some machine-read finding, so
  the subset is too small to test in any variant (see the counts above). Normal-read
  evidence therefore comes from EchoNext alone; `BEYOND` has its per-dataset numbers.

## Confirmatory replication on val

Everything above chose candidates on **test**. This section checks them once on the
**val** split, which no part of PILA uses (assignment, projection and the linear probe
all fit on train) and which none of the analysis choices above looked at.

**Fixed before any val result was computed** (git history shows this section was
committed before the val outputs existed):

- **Prototypes tested** (`CONFIRMATORY`): for LV systolic dysfunction, 991 (met every
  criterion on test) and 850 (met them before the spline adjustment; Holm p 0.07 in
  MIMIC after it). Nothing for aortic valve disease, where nothing qualified.
- **Analysis**: exactly the test analysis under the main variant: **+ imputed reads**,
  spline interval terms, the known-findings model fit on train, and the cutoffs and
  reliable findings from MIMIC train. The cell below asserts these settings.
- **Primary criterion**, per prototype: adjusted OR > 1 with Holm-corrected p < 0.05
  in **both** datasets. Holm runs over the 4 tests (2 prototypes × 2 datasets).
- **Secondary, reported but not part of the criterion**: ΔAUROC of the candidates
  over known findings, and the normal-read result wherever the subset has at least
  `MIN_NORMAL_POS` positives (EchoNext, in practice).

The val inputs come from `scripts/8-z-run-protossl-heedb-pila-val.sh`. Until both
datasets' outputs are published, the cells below only report that they're missing.

In [ ]:
CONFIRMATORY = {"LV systolic dysfunction": [991, 850]}
CONFIRMATORY_ALPHA = 0.05
VAL_RUN = IMPUTED_RUN
VAL_EXPS = {
    "embeds": "protossl-heedb-pila-val",
    "imputed": "protossl-heedb-pila-from-ptbxl-val",
}

# the analysis is fixed: fail loudly if a setting it depends on has changed
assert (MAIN_VARIANT, INTERVAL_TERMS) == ("+ imputed reads", "spline")
assert (MIN_IMPUTED_AUROC, MIN_NORMAL_POS) == (0.8, 20)
for label, protos in CONFIRMATORY.items():
    assert set(protos) <= shared_prototypes(label), label


def val_dir(name: str, kind: str) -> Path:
    slug = ALL_DATASETS[name]["slug"]
    return (
        SHARED_ROOT / "runs" / VAL_RUN / f"protossl-outputs-seed{SEED}" / f"runs-{slug}"
        / VAL_EXPS[kind]
    )


VAL_READY = all(
    (val_dir(name, "embeds") / "compute-embeddings/latest/val_embeds.npy").exists()
    and (val_dir(name, "imputed") / "val_probs.npy").exists()
    for name in DATASETS
)
if VAL_READY:
    for name in DATASETS:
        slug = ALL_DATASETS[name]["slug"]
        sources = {kind: (val_dir(name, kind) / "SOURCE").read_text().strip() for kind in VAL_EXPS}
        assert sources["embeds"].endswith(f"seed{SEED}/runs-{slug}/{MODEL}"), sources
        assert sources["imputed"].endswith(f"seed{SEED}/runs-ptbxl/{MODEL}"), sources
        SPLITS[name, "val"] = load_split(
            name,
            "val",
            embeds_dir=val_dir(name, "embeds") / "compute-embeddings/latest",
            imputed_from=val_dir(name, "imputed"),
        )
    check_alignment("val")
else:
    print(
        "val outputs not published yet, run scripts/8-z-run-protossl-heedb-pila-val.sh "
        f"for {', '.join(ALL_DATASETS[n]['slug'] for n in DATASETS)} (run {VAL_RUN})"
    )

In [ ]:
def confirm() -> tuple[pd.DataFrame, pd.DataFrame]:
    """The fixed val analysis: (per prototype and dataset, verdict per prototype)."""
    gains, beyond = beyond_known(MAIN_VARIANT, eval_split="val", prototypes=CONFIRMATORY)
    display(gains.round(3))
    wide = beyond.unstack("dataset")
    confirmed = (
        (wide["OR adjusted"] > 1) & (wide["p adjusted (Holm)"] < CONFIRMATORY_ALPHA)
    ).all(axis=1)
    return beyond, confirmed.rename("confirmed").to_frame()


if VAL_READY:
    VAL_BEYOND, VAL_CONFIRMED = confirm()
    display(VAL_BEYOND.round(3))
    display(VAL_CONFIRMED)

## Average beats for review

For the cardiologist's reading: what the ECGs a prototype responds to look like on
average, against otherwise similar ECGs it doesn't respond to. A single projected
window shows one example; an average over many shows the consistent pattern.

Per prototype and dataset, on **val**:

- **High-activation**: the `REVIEW_N` normal-read ECGs (main variant, so no machine
  or reliably imputed finding and normal intervals) with the highest similarity to
  the prototype.
- **Matched low-activation**: for each of those, the nearest normal-read ECG from the
  lower half of similarity with the same sex, by age, heart rate and QRS duration
  (`MATCH_ON`, standardized), without replacement. Matching on QRS keeps the
  comparison from being a QRS-width difference.
- **Beat**: each ECG contributes the beat whose R peak (largest summed absolute
  amplitude) falls inside the 1-second window the prototype matched, cut
  `BEAT_WINDOW` around it. ECGs whose beat runs off the recording are dropped.
- Not matched on PR: compare the PR column of the balance table at the end.
- Lines are group means and bands their 95% CIs (±1.96 SE). All leads share one
  y-scale. MIMIC is in mV; EchoNext ships normalized waveforms, so its units are
  EchoNext's own.

`REVIEW_PROTOTYPES` holds the two candidates and two known-pattern controls (217,
PTB-XL ventricular aneurysm; 728, LVH), so the reading can check whether the
candidates look different from known patterns. Each figure is saved twice: labeled,
and **blinded** (neutral pattern and dataset codes, no prototype, label or PTB-XL
reading), with the key in `FIG_DIR / "review-blinding-key.csv"` for whoever runs
the reading.

In [ ]:
from protossl.plotting import LEAD_LAYOUT, LEAD_TO_IDX

# prototype -> label it was shared for: the candidates, then known-pattern controls
REVIEW_PROTOTYPES = {
    850: "LV systolic dysfunction",
    991: "LV systolic dysfunction",
    217: "LV systolic dysfunction",
    728: "Aortic valve disease",
}
REVIEW_N = 100
MATCH_ON = ["age", "heart rate", "QRS"]  # plus exact sex
BEAT_WINDOW = (-0.3, 0.5)  # seconds around the R peak
REVIEW_COLORS = {"high": "#2a78d6", "low": "#8a8984"}  # series slot 1, neutral gray

for proto, label in REVIEW_PROTOTYPES.items():
    assert proto in shared_prototypes(label), (proto, label)


def prototype_column(name: str, proto: int) -> int:
    """Embedding column of `proto`: the slot `name` assigned it to."""
    assign = ALL_DATASETS[name]["assign"][SEED]
    slots = assign.index[assign["prototype_idx"] == proto]
    assert len(slots) == 1, (name, proto)
    return int(slots[0])


def match_controls(cases: np.ndarray, pool: np.ndarray, cov: pd.DataFrame) -> np.ndarray:
    """Nearest pool row per case (same sex, standardized MATCH_ON), no replacement."""
    X = cov[MATCH_ON].to_numpy(float)
    X = (X - X[pool].mean(0)) / X[pool].std(0)
    sex = cov["sex"].to_numpy()
    available = set(pool.tolist())
    controls = []
    for case in cases:  # most-activated first, so they get the closest matches
        candidates = np.array([j for j in available if sex[j] == sex[case]])
        best = candidates[np.argmin(((X[candidates] - X[case]) ** 2).sum(1))]
        controls.append(best)
        available.remove(best)
    return np.array(controls)


def review_groups(name: str, proto: int) -> tuple[np.ndarray, np.ndarray]:
    """Val row indices of the high-activation ECGs and their matched controls."""
    data = SPLITS[name, "val"]
    _, normal = KNOWN_VARIANTS[MAIN_VARIANT](data)
    sim = data["embeds"][:, prototype_column(name, proto)]
    rows = np.flatnonzero(normal)
    high = rows[np.argsort(-sim[rows])[:REVIEW_N]]
    pool = rows[sim[rows] < np.median(sim[rows])]
    return high, match_controls(high, pool, data["cov"])


def aligned_beats(name: str, proto: int, rows: np.ndarray) -> np.ndarray:
    """(n, L, T) beats centered on the R peak inside each ECG's matched window."""
    dataset, data = get_dataset(name, "val"), SPLITS[name, "val"]
    chunks = data["chunks"][:, prototype_column(name, proto)]
    fs = SAMPLING_RATE
    before, after = int(-BEAT_WINDOW[0] * fs), int(BEAT_WINDOW[1] * fs)
    beats = []
    for row in rows:
        signal = get_ecg_waveform(dataset=dataset, index=int(row), sampling_rate=fs)
        start, end = chunk_to_seconds(
            int(chunks[row]), chunk_len=fs, chunk_overlap=0.5, sampling_rate=fs
        )
        lo, hi = int(start * fs), int(end * fs)
        r_peak = lo + int(np.argmax(np.abs(signal[:, lo:hi]).sum(axis=0)))
        if r_peak - before >= 0 and r_peak + after <= signal.shape[1]:
            beats.append(signal[:, r_peak - before : r_peak + after])
    return np.stack(beats)


def balance(name: str, label: str, high: np.ndarray, low: np.ndarray) -> pd.DataFrame:
    """Group means of the matching and other covariates, and the label's prevalence."""
    data = SPLITS[name, "val"]
    cov = data["cov"][["age", "sex", "heart rate", "PR", "QRS", "QTc"]]
    y = data["labels"][:, list(ALL_DATASETS[name]["targets"]).index(LABELS[label][name])]
    table = pd.DataFrame({"high-activation": cov.iloc[high].mean(), "matched": cov.iloc[low].mean()})
    table.loc[f"{label} prevalence"] = [y[high].mean(), y[low].mean()]
    return table

In [ ]:
def plot_average_beats(
    high: np.ndarray, low: np.ndarray, units: str, title: str
) -> plt.Figure:
    """Mean beat per lead with 95% CI bands, high-activation over matched controls."""
    fs = SAMPLING_RATE
    t = np.arange(int(BEAT_WINDOW[0] * fs), int(BEAT_WINDOW[1] * fs)) / fs
    groups = [  # drawn in this order, so the high-activation line sits on top
        ("low", low, "--", f"matched low-activation (n={len(low)})"),
        ("high", high, "-", f"high-activation (n={len(high)})"),
    ]
    with plt.rc_context({"font.size": 7}):
        fig, axs = plt.subplots(3, 4, sharex=True, sharey=True, figsize=(TEXT_WIDTH, 3.6))
        for r, row in enumerate(LEAD_LAYOUT):
            for c, lead in enumerate(row):
                ax, i = axs[r, c], LEAD_TO_IDX[lead]
                # ECG-paper time grid, kept recessive: 0.04 s minor, 0.2 s major
                for x in np.arange(-0.28, BEAT_WINDOW[1], 0.04):
                    ax.axvline(x, color="#f3d6d6", linewidth=0.3, zorder=0)
                for x in np.arange(-0.2, BEAT_WINDOW[1], 0.2):
                    ax.axvline(x, color="#e8b4b4", linewidth=0.5, zorder=0)
                ax.axhline(0, color="#d9d8d4", linewidth=0.6, zorder=0)
                for key, beats, style, _ in groups:
                    mean = beats[:, i].mean(axis=0)
                    ci = 1.96 * beats[:, i].std(axis=0, ddof=1) / np.sqrt(len(beats))
                    ax.fill_between(t, mean - ci, mean + ci, color=REVIEW_COLORS[key], alpha=0.2, linewidth=0)
                    ax.plot(t, mean, color=REVIEW_COLORS[key], linestyle=style, linewidth=1.2)
                ax.text(0.03, 0.95, lead, transform=ax.transAxes, ha="left", va="top", fontweight="bold")
                ax.set_xlim(t[0], t[-1])
                for side in ["top", "right"]:
                    ax.spines[side].set_visible(False)
                ax.tick_params(length=2)
        for ax in axs[:, 0]:
            ax.set_ylabel(units)
        for ax in axs[-1]:
            ax.set_xlabel("s from R peak")
        handles = [
            plt.Line2D([], [], color=REVIEW_COLORS[key], linestyle=style, linewidth=1.2, label=name)
            for key, _, style, name in reversed(groups)
        ]
        fig.legend(handles=handles, loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 0))
        fig.suptitle(title, fontsize=8, fontweight="bold")
        fig.tight_layout(rect=(0, 0.06, 1, 1))
    return fig

In [ ]:
# blinding: neutral codes, shuffled so the order doesn't reveal which are candidates
_rng = np.random.default_rng(20261119)
PATTERN_CODES = dict(zip(REVIEW_PROTOTYPES, _rng.permutation(list("ABCD"))))
DATASET_CODES = dict(zip(DATASETS, _rng.permutation([1, 2]).tolist()))
UNITS = {"EchoNext": "EchoNext units", "MIMIC": "mV"}

if VAL_READY:
    key, balances = [], {}
    for proto, label in REVIEW_PROTOTYPES.items():
        reading = INTERPRETATIONS[label].loc[proto, "PTB-XL"]
        for name in DATASETS:
            high, low = review_groups(name, proto)
            balances[proto, name] = balance(name, label, high, low)
            beats_high, beats_low = aligned_beats(name, proto, high), aligned_beats(name, proto, low)
            labeled = plot_average_beats(
                beats_high, beats_low, UNITS[name],
                f"Prototype {proto} (PTB-XL: {reading}) | {name} val | shared for {label}",
            )
            stem = f"{LABEL_SLUGS[label]}-p{proto}-avgbeat-{ALL_DATASETS[name]['slug']}-val-seed{SEED}"
            labeled.savefig(FIG_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
            labeled.savefig(FIG_DIR / f"{stem}.pdf", bbox_inches="tight")
            plt.show()

            code, ds_code = PATTERN_CODES[proto], DATASET_CODES[name]
            blinded = plot_average_beats(
                beats_high, beats_low, "amplitude", f"Pattern {code}, dataset {ds_code}"
            )
            blind_stem = f"review-pattern-{code}-dataset{ds_code}"
            blinded.savefig(FIG_DIR / f"{blind_stem}.png", dpi=300, bbox_inches="tight")
            blinded.savefig(FIG_DIR / f"{blind_stem}.pdf", bbox_inches="tight")
            plt.close(blinded)
            key.append(
                {"figure": blind_stem, "pattern": code, "dataset code": ds_code, "prototype": proto,
                 "dataset": name, "shared for": label, "PTB-XL reading": reading}
            )
    pd.DataFrame(key).to_csv(FIG_DIR / "review-blinding-key.csv", index=False)

In [ ]:
# matching balance and label prevalence per group (not for the blinded reading)
if VAL_READY:
    display(pd.concat({f"{p} {n}": t for (p, n), t in balances.items()}, axis=1).round(2).T)

## Exploratory: known findings from outside ProtoSSL

**Not prespecified.** This section was added after the val results, following a
review of the notebook. Its val numbers are a second look, not a replication; the
confirmatory result is the section above.

The main variant leaves two gaps:

- **Some present/absent reads catch little of their finding.** Imputed incomplete
  LBBB flags 0.4% of MIMIC ECGs and catches 4% of the machine-read ones on train (0%
  on test), and 991 is one of PTB-XL's incomplete LBBB prototypes. Low voltage is
  recovered at AUROC 0.80; poor R-wave progression has no PTB-XL code.
- **Graded probabilities can't take over,** because they absorb every prototype
  (measured below).

Here known findings also come from **ECGFounder**, a foundation model trained to
predict HEEDB's machine-read statements, outside ProtoSSL's feature space. Its
150-output statement head is applied to its embeddings: `5-1`'s `ecgfounder-logreg`
for train and test, `scripts/8-y-run-ecgfounder-val.sh` for val. The head's statement
names and order are `tasks.txt` from the ECGFounder repo (commit `04edac7`, read the
same way by their `ptbxl_eval.py`), saved as `scripts/ecg-fms/ecgfounder_tasks.txt`.

1. Check that the embeddings line up with our rows and that the head's outputs mean
   what their names say, against MIMIC's machine reads.
2. Measure how much of each prototype's similarity each set of graded known findings
   can rebuild.
3. Rerun the per-prototype test under graded known findings, on test and val, for
   every shared LV systolic dysfunction prototype. The non-candidates serve as
   controls.

In [ ]:
import hashlib

import joblib
import torch
from sklearn.linear_model import LinearRegression

EF_CKPT = Path("/opt/gpudata/models/PKUDigitalHealth/ECGFounder/12_lead_ECGFounder.pth")
EF_SHA256 = "ee199f3781f4ae1f732973267f003da0a759ea12bddb0dd28a77faa60aca7997"  # scripts/_common.sh
EF_TASKS_FILE = Path("../scripts/ecg-fms/ecgfounder_tasks.txt")
EF_EXP = "ecgfounder-logreg"  # 5-1: train/test embeddings and a probe of the dataset's own labels
EF_VAL_EXP = "ecgfounder-val"  # 8-y: val embeddings

assert hashlib.sha256(EF_CKPT.read_bytes()).hexdigest() == EF_SHA256
EF_TASKS = EF_TASKS_FILE.read_text().splitlines()
# weights_only=False as in scripts/ecg-fms/_compute_ecgfounder_embeddings.py (pinned by sha256)
_head = torch.load(EF_CKPT, map_location="cpu", weights_only=False)["state_dict"]
EF_W, EF_B = _head["dense.weight"].numpy(), _head["dense.bias"].numpy()
del _head
assert len(EF_TASKS) == len(set(EF_TASKS)) == len(EF_B) == 150


def ef_val_dir(name: str) -> Path:
    slug = ALL_DATASETS[name]["slug"]
    return SHARED_ROOT / "runs" / VAL_RUN / f"protossl-outputs-seed{SEED}" / f"runs-{slug}" / EF_VAL_EXP


EF_VAL_READY = VAL_READY and all((ef_val_dir(n) / "val_embeds.npy").exists() for n in DATASETS)
EF_SPLITS = ["train", "test"] + (["val"] if EF_VAL_READY else [])

for name in DATASETS:
    exp = run_dir(ALL_DATASETS[name], SEED, EF_EXP)
    own_probe = joblib.load(exp / "model.joblib")
    scaler = StandardScaler().fit(np.load(exp / "train_embeds.npy"))  # as _linear_probe.py did
    k = list(ALL_DATASETS[name]["targets"]).index(LABELS["LV systolic dysfunction"][name])
    for split in EF_SPLITS:
        emb = np.load((ef_val_dir(name) if split == "val" else exp) / f"{split}_embeds.npy")
        data = SPLITS[name, split]
        assert len(emb) == len(data["labels"]), (name, split)
        data["ef"] = pd.DataFrame(emb @ EF_W.T + EF_B, columns=[f"ECGFounder {t}" for t in EF_TASKS])
        # rows follow ours: 5-1's own-label probe reproduces its test probs, and
        # predicts our LV systolic dysfunction labels on every split
        probs = np.asarray([p[:, 1] for p in own_probe.predict_proba(scaler.transform(emb))]).T
        if split == "test":
            assert np.abs(probs - np.load(exp / "probs.npy")).max() < 1e-6, name
        assert roc_auc_score(data["labels"][:, k], probs[:, k]) > 0.8, (name, split)

if not EF_VAL_READY:
    print(
        "ECGFounder val embeddings not published yet, run scripts/8-y-run-ecgfounder-val.sh "
        f"for {', '.join(ALL_DATASETS[n]['slug'] for n in DATASETS)} (run {VAL_RUN})"
    )
print("ECGFounder statements on:", EF_SPLITS)

In [ ]:
# known finding -> ECGFounder statements that express it (its score is their maximum
# logit). No ECGFounder statement is R-wave progression.
EF_FINDINGS = {
    "infarct / Q waves": [t for t in EF_TASKS if "INFARCT" in t and "ACUTE" not in t],
    "LBBB": ["LEFT BUNDLE BRANCH BLOCK"],
    "incomplete LBBB": ["INCOMPLETE LEFT BUNDLE BRANCH BLOCK"],
    "RBBB": ["RIGHT BUNDLE BRANCH BLOCK", "INCOMPLETE RIGHT BUNDLE BRANCH BLOCK"],
    "IVCD": ["NONSPECIFIC INTRAVENTRICULAR CONDUCTION DELAY", "NONSPECIFIC INTRAVENTRICULAR BLOCK"],
    "fascicular block": ["LEFT ANTERIOR FASCICULAR BLOCK"],
    "LVH": ["LEFT VENTRICULAR HYPERTROPHY", "VOLTAGE CRITERIA FOR LEFT VENTRICULAR HYPERTROPHY"],
    "left atrial abnormality": ["LEFT ATRIAL ENLARGEMENT"],
    "low voltage": ["LOW VOLTAGE QRS"],
    "ST-T abnormality": [
        "NONSPECIFIC T WAVE ABNORMALITY",
        "NONSPECIFIC ST ABNORMALITY",
        "NONSPECIFIC ST AND T WAVE ABNORMALITY",
        "WITH REPOLARIZATION ABNORMALITY",
    ],
    "AF / flutter": ["ATRIAL FIBRILLATION", "ATRIAL FLUTTER"],
    "paced": ["ELECTRONIC VENTRICULAR PACEMAKER", "VENTRICULAR-PACED RHYTHM", "ATRIAL-PACED RHYTHM"],
    "ectopy": ["PREMATURE VENTRICULAR COMPLEXES", "PREMATURE ATRIAL COMPLEXES"],
    "AV block": ["WITH 1ST DEGREE AV BLOCK", "WITH PROLONGED AV CONDUCTION"],
}
assert all(t in EF_TASKS for ts in EF_FINDINGS.values() for t in ts)
assert set(EF_FINDINGS) <= set(KNOWN_STATEMENTS)


def validate_ef(split: str = "test") -> pd.DataFrame:
    """AUROC of each graded known finding against MIMIC's machine read."""
    data = SPLITS["MIMIC", split]
    rows = []
    for finding in KNOWN_STATEMENTS:
        machine = data["cov"][finding].to_numpy().astype(bool)
        row = {"finding": finding, "machine-read prevalence": machine.mean()}
        if finding in EF_FINDINGS:
            score = data["ef"][[f"ECGFounder {t}" for t in EF_FINDINGS[finding]]].max(axis=1)
            row["ECGFounder AUROC"] = roc_auc_score(machine, score)
        if finding in IMPUTED_FINDINGS:
            row["PTB-XL probe AUROC"] = roc_auc_score(machine, data["imputed_scores"][finding])
        rows.append(row)
    return pd.DataFrame(rows).set_index("finding")


validate_ef().round(3)

### How much of a prototype graded known findings contain

The linear R² (fit on train, scored on test) of each shared prototype's similarity,
rebuilt from:

- the 71 imputed PTB-XL log-odds (what + imputed probabilities adjusts for);
- PTB-XL's prototype similarities, the inputs of that probe, without the tested
  prototype's own slot;
- ECGFounder's 150 statement logits.

Graded known findings that rebuild nearly all of every prototype absorb any
prototype's effect, so failing under them says nothing about a particular prototype.
Dropping the tested prototype's slot from PTB-XL's probe (alone, or with every slot
correlated with it at |r| ≥ 0.5) changed no adjusted OR at two decimals; that refit
isn't repeated here, the R² below shows why.

In [ ]:
def rebuild_r2(label: str = "LV systolic dysfunction") -> pd.DataFrame:
    """R^2 of each shared prototype's similarity from each set of graded known findings."""
    ptbxl_slots = ALL_DATASETS["PTB-XL"]["assign"][SEED]["prototype_idx"].to_numpy()
    rows = []
    for name in DATASETS:
        assign = ALL_DATASETS[name]["assign"][SEED]
        train, test = SPLITS[name, "train"], SPLITS[name, "test"]
        # PTB-XL's prototype similarities on this dataset (one column per PTB-XL slot)
        sims = {
            s: np.load(imputed_dir(name) / f"compute-embeddings/latest/{s}_embeds.npy")
            for s in ["train", "test"]
        }
        assert sims["test"].shape == (len(test["labels"]), len(ptbxl_slots)), name
        for proto in sorted(shared_prototypes(label)):
            col = int(assign.index[assign["prototype_idx"] == proto][0])
            others = ptbxl_slots != proto  # all slots if PTB-XL doesn't use the prototype
            sources = {
                "imputed PTB-XL log-odds": (train["imputed"], test["imputed"]),
                "PTB-XL prototypes, minus this one": (sims["train"][:, others], sims["test"][:, others]),
                "ECGFounder statements": (train["ef"], test["ef"]),
            }
            row = {"dataset": name, "prototype": proto}
            for source, (X_train, X_test) in sources.items():
                fit = LinearRegression().fit(X_train, train["embeds"][:, col])
                row[source] = fit.score(X_test, test["embeds"][:, col])
            rows.append(row)
    return pd.DataFrame(rows).set_index(["dataset", "prototype"])


rebuild_r2().round(2)

### The candidates under graded known findings

Each variant uses the main variant's models, splines and normal-read subset:

- **+ reads + graded conduction/voltage (PTB-XL probe)**: the main variant plus the
  imputed log-odds of ILBBB, CLBBB, IVCD, LVOLT and LAFB. These are 991's classical
  readings, without the rest of the probe.
- **+ reads + graded conduction/voltage/axis (ECGFounder)**: the main variant plus
  ECGFounder's incomplete LBBB, LBBB, IVCD, low voltage, LAFB and axis deviation
  statements. The axis statements also stand in for EchoNext's missing axis
  measurement.
- **+ ECGFounder statements**: as recorded, plus all 150 ECGFounder logits.
- **+ ECGFounder statements + reads**: the main variant plus all 150.

The tables give the adjusted OR per SD with its nominal p; `EXPLORATORY_BEYOND` also
has Holm within each split and variant (14 tests).

In [ ]:
GRADED_PTBXL = [f"PTB-XL {c}" for c in ["ILBBB", "CLBBB", "IVCD", "LVOLT", "LAFB"]]
GRADED_EF = [
    f"ECGFounder {t}"
    for t in [
        "INCOMPLETE LEFT BUNDLE BRANCH BLOCK",
        "LEFT BUNDLE BRANCH BLOCK",
        "NONSPECIFIC INTRAVENTRICULAR CONDUCTION DELAY",
        "LOW VOLTAGE QRS",
        "LEFT ANTERIOR FASCICULAR BLOCK",
        "LEFT AXIS DEVIATION",
        "RIGHT AXIS DEVIATION",
        "RIGHTWARD AXIS",
    ]
]
_main = KNOWN_VARIANTS[MAIN_VARIANT]


def _main_plus(extra):
    """The main variant's known findings and normal-read subset, plus `extra(data)`."""

    def get_known(data):
        cov, normal = _main(data)
        return pd.concat([cov, extra(data)], axis=1), normal

    return get_known


EXPLORATORY_VARIANTS = {
    "+ reads + graded conduction/voltage (PTB-XL probe)": _main_plus(lambda d: d["imputed"][GRADED_PTBXL]),
    "+ reads + graded conduction/voltage/axis (ECGFounder)": _main_plus(lambda d: d["ef"][GRADED_EF]),
    "+ ECGFounder statements": lambda d: (pd.concat([d["cov"], d["ef"]], axis=1), _main(d)[1]),
    "+ ECGFounder statements + reads": _main_plus(lambda d: d["ef"]),
}
COMPARED = [MAIN_VARIANT, "+ imputed probabilities", *EXPLORATORY_VARIANTS]
_lvsd = {"LV systolic dysfunction": sorted(shared_prototypes("LV systolic dysfunction"))}

_gains, _beyond = {}, {}
for split in [s for s in EF_SPLITS if s != "train"]:
    for variant in COMPARED:
        _gains[split, variant], _beyond[split, variant] = beyond_known(
            variant,
            eval_split=split,
            prototypes=_lvsd,
            variants=KNOWN_VARIANTS | EXPLORATORY_VARIANTS,
        )
EXPLORATORY_GAINS = pd.concat(_gains, names=["split", "known findings"])
EXPLORATORY_BEYOND = pd.concat(_beyond, names=["split", "known findings"])


def or_table(dataset: str) -> pd.DataFrame:
    """Adjusted OR per SD (nominal p), prototypes by known-findings variant."""
    t = EXPLORATORY_BEYOND.xs(dataset, level="dataset")
    cell = t["OR adjusted"].map("{:.2f}".format) + " (" + t["p adjusted"].map("{:.3f}".format) + ")"
    return cell.unstack("known findings")[COMPARED].droplevel("label")


for name in ["MIMIC", "EchoNext"]:
    print(name)
    display(or_table(name))
EXPLORATORY_GAINS[["ΔAUROC", "95% CI lo", "95% CI hi"]].unstack("dataset").round(3)

### Reading the exploratory results

At `SEED = 42`; update by hand on reruns.

- **ECGFounder's head reads what its names say.** Against MIMIC's machine reads
  (test), it recovers incomplete LBBB at AUROC 0.94, IVCD 0.86, left atrial
  enlargement 0.88 and low voltage 0.83, all better than PTB-XL's probe. Its LVH
  (0.71) and infarct (0.81) are weaker, which is why one variant keeps the reads
  alongside it. On EchoNext it can't be checked (no machine reads).
- **The graded-probability null isn't evidence about any one prototype.** The 71
  imputed log-odds rebuild 86–92% of every shared prototype's similarity, and
  PTB-XL's other prototypes 92–96%, whereas ECGFounder's statements rebuild 46–66%.
  Under + imputed probabilities every prototype moves toward 1, candidates and
  controls alike (0.95–1.16 on val).
- **991 holds up in MIMIC.** On test, every ECGFounder variant leaves it at 1.48–1.60
  (p ≤ 0.001). On val it stays at 1.21–1.22 (p ≈ 0.03) with graded
  conduction/voltage from either source, 1.18 (p 0.064) with all ECGFounder
  statements, and 1.15 (p 0.11) with those plus the reads. So graded incomplete LBBB,
  LBBB, IVCD and low voltage don't explain it, and adjusting for everything ECGFounder
  reads shrinks it to a size val can't confirm.
- **850 is mostly graded conduction/voltage.** In MIMIC its OR drops to 1.14 (test)
  and 1.07 (val) with PTB-XL's graded conduction/voltage scores. With all ECGFounder
  statements it is 1.14–1.15 on val. In EchoNext its QRS is lower in nearly every lead (see
  the average beats), which fits low voltage.
- **The controls behave as controls in MIMIC:** 186, 217, 522, 578 and 617 are null
  or negative under every graded variant. The one exception anywhere is 217 on val
  under the main variant (1.19, p 0.039). In EchoNext, 522, 578 and 617 stay at about
  1.2–1.4 even with all ECGFounder statements, so EchoNext still can't tell
  candidates from controls.
- **Still not adjusted for:** poor R-wave progression outside MIMIC's machine read.
  The analysis is also single seed and post hoc.